imports

In [1]:
import pandas as pd
import glob
import os


It uses Python’s glob module to search through every folder and sub‑folder in the project directory.

It looks for files whose names match specific patterns:

*-street.csv → street‑level crime data files

*-outcomes.csv → crime outcome data files

recursive=True ensures the search includes all nested directories.

The result is two lists:

street_files — paths to all street‑level crime datasets

outcome_files — paths to all outcome datasets

This approach is useful when working with large datasets that are split across multiple monthly or yearly files.
Data contained in the street‑level files (*-street.csv)
These files typically contain one row per recorded crime, including:

A unique crime ID

The month and police force

Crime type (e.g., burglary, violence, theft)

Location information (LSOA, latitude/longitude)

Whether the location was anonymised

Basic contextual details about the incident

These datasets represent the raw crime events.
Data contained in the outcomes files (*-outcomes.csv)
These files describe what happened after the crime was recorded, including:

The same crime ID (used to link back to the street file)

The month and police force

The outcome category (e.g., “Under investigation”, “No suspect identified”, “Suspect charged”)

These datasets provide the follow‑up actions taken by police.

In [2]:
street_files = glob.glob("./**/*-street.csv", recursive=True)
outcome_files = glob.glob("./**/*-outcomes.csv", recursive=True)

This line loads all outcome and street CSV files and merges them into a single, clean DataFrame ready for analysis.

In [3]:
street_df = pd.concat((pd.read_csv(f) for f in street_files), ignore_index=True)
outcome_df = pd.concat((pd.read_csv(f) for f in outcome_files), ignore_index=True)

determining the structuer

In [4]:
#outcome
outcome_df.info()
outcome_df.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2920000 entries, 0 to 2919999
Data columns (total 10 columns):
 #   Column        Dtype  
---  ------        -----  
 0   Crime ID      object 
 1   Month         object 
 2   Reported by   object 
 3   Falls within  object 
 4   Longitude     float64
 5   Latitude      float64
 6   Location      object 
 7   LSOA code     object 
 8   LSOA name     object 
 9   Outcome type  object 
dtypes: float64(2), object(8)
memory usage: 222.8+ MB


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Outcome type
0,e654e620cf71a660f56f13517cadb7862c9fd12c4bf63f...,2023-02,Avon and Somerset Constabulary,Avon and Somerset Constabulary,NaN,NaN,No location,NaN,NaN,Unable to prosecute suspect
1,8b96302723123488b3dcf9a7821656a18904869710827c...,2023-02,Avon and Somerset Constabulary,Avon and Somerset Constabulary,NaN,NaN,No location,NaN,NaN,Suspect charged


In [5]:
#street
street_df.info()
street_df.head(2)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3651324 entries, 0 to 3651323
Data columns (total 12 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   Crime ID               object 
 1   Month                  object 
 2   Reported by            object 
 3   Falls within           object 
 4   Longitude              float64
 5   Latitude               float64
 6   Location               object 
 7   LSOA code              object 
 8   LSOA name              object 
 9   Crime type             object 
 10  Last outcome category  object 
 11  Context                float64
dtypes: float64(3), object(9)
memory usage: 334.3+ MB


,Crime ID,Month,Reported by,Falls within,Longitude,Latitude,Location,LSOA code,LSOA name,Crime type,Last outcome category,Context
0,44949804e10105361f806c22620b183ed7b4337bc53c62...,2023-02,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-1.563651,53.550385,On or near Hawthorn Grove,E01007420,Barnsley 016C,Vehicle crime,Status update unavailable,NaN
1,NaN,2023-02,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-2.491146,51.425008,On or near Maximus Gardens,E01014399,Bath and North East Somerset 001A,Anti-social behaviour,NaN,NaN


changing the headings to ensure matching

In [6]:
street_df.columns = (
    street_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

outcome_df.columns = (
    outcome_df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

spliting the month collomn that conatins both year and month

In [7]:
street_df["year"] = street_df["month"].str[:4].astype(int)
street_df["month_num"] = street_df["month"].str[5:7].astype(int)

outcome_df["year"] = outcome_df["month"].str[:4].astype(int)
outcome_df["month_num"] = outcome_df["month"].str[5:7].astype(int)

ensuring correct import

In [8]:
display(street_df.head(1),outcome_df.head(1))

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,context,year,month_num
0,44949804e10105361f806c22620b183ed7b4337bc53c62...,2023-02,Avon and Somerset Constabulary,Avon and Somerset Constabulary,-1.563651,53.550385,On or near Hawthorn Grove,E01007420,Barnsley 016C,Vehicle crime,Status update unavailable,NaN,2023,2


,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,outcome_type,year,month_num
0,e654e620cf71a660f56f13517cadb7862c9fd12c4bf63f...,2023-02,Avon and Somerset Constabulary,Avon and Somerset Constabulary,NaN,NaN,No location,NaN,NaN,Unable to prosecute suspect,2023,2


These lines are used to check whether the crime_id field in the street‑level dataset behaves as a proper unique identifier. The first line, street_df["crime_id"].is_unique, returns a Boolean value indicating whether every crime record has a distinct ID. The second line, street_df["crime_id"].duplicated().sum(), counts how many IDs appear more than once by identifying duplicated entries and summing them. Together, these checks help verify data quality and ensure that crime_id can be safely used for merging datasets or analysing individual crime incidents.

In [9]:
street_df["crime_id"].is_unique
street_df["crime_id"].duplicated().sum()

np.int64(610586)

The value 610586 represents the number of duplicated crime_id entries detected in the raw street‑level dataset. This showed that many crime records were repeated across files, which is common in UK Police data due to overlapping monthly downloads and multiple outcomes linked to the same incident. As part of the cleaning process, these repeated entries were removed so that each crime appears only once in the final dataset. This ensures the data is not inflated by duplicates and provides a reliable foundation for calculating crime rates and performing further analysis.

This code identifies which crime_id values appear most frequently in the dataset by counting how many times each ID occurs and sorting the results in descending order. The output, stored in dupe_counts, shows the crime IDs with the highest number of repetitions at the top. This is useful for understanding the scale and pattern of duplication in the raw data before cleaning, helping confirm which IDs were repeated and how often they appeared prior to removing duplicates.

In [10]:
dupe_counts = (
    street_df["crime_id"]
    .value_counts()
    .sort_values(ascending=False)
)
dupe_counts

crime_id
6198ccbb0df050a7570ef4976ac9c0029993dec216416a6648e1db4b496eb53b    157
a30209f2ff7df17a641b9ef88dcb7e865421b26dfa96f33583353f572e5fa5bd      8
fa1cff6c25fe6c5d016168877f33f55e2db84a9d74f5387e2251fb6d2e2bc6c2      7
08ded05a1fb9b53c1e3e320d41e161ad9c427ea880f63a2a76e2451e25807ece      6
ea659bdd4ed38ac5c9259bb280638a9ab0a1650137f57e2a7efbfe080e285d87      6
                                                                   ... 
e56425a27a97dcd3c1b034e46ca68fcce1ea2db477d3a5bd26be47913c84e3d4      1
d458d04cc78d718859c12879d3c53736d62e386489fa3117557ead3bc5b6de0a      1
d9c1cca5a0570a3a6cf9c5c3afd06caa0ec6f75938728938f23ef0108b913f11      1
aabcc5a5420bc9c71b410c4c38d26c88186c43a5f7cda1c3bf5a764f84a7e0aa      1
890774c71e82c68efc90af4e1cf0624cfc99f31d11287640a1a972e611bafbf5      1
Name: count, Length: 3040737, dtype: int64

In [11]:
dupe_counts2 = (
    outcome_df["crime_id"]
    .value_counts()
    .sort_values(ascending=False)
)
dupe_counts2

crime_id
89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a67755c99d4c32f25a0    45
31410185a0d3702c584208a38621f9ea8c77ac62ad3d91f42add3fad542fc99d    44
41ed9838d946e7d20ba35ad3c4887acc3622d7ca486dbae8e724919ac3d7ddf2    44
7d31f911cc3c4836a88061a3173c964c07b3e1e344add8fc76ba59cae482d137    30
6ecafd31e42b0d9cf2250a6945f801fde43963aa4312e3714acb42e631f84f2d    30
                                                                    ..
e8cd9813ea716609af70b37657eff52dda0ce909f4a32b09aa3dbe65b8d5be04     1
15edaa75218e0576a21312a53d76ef40f41f9395fdbdb305577e27e4edebadfe     1
33a3af6bc7da6fa3ab4553d9c586d3e3d1ea2e0f197a085c405577cd1ef05a4b     1
921b47c5604d8f8ca704fa39dea2080286685bd50201fd00199c88edf181d6f2     1
5fed7ba966ead190bab135cdac58b9caefbf73a6de3f757769e872aac27b6ed8     1
Name: count, Length: 2617845, dtype: int64

checking out a duplicate crime ID

In [12]:
rows = outcome_df[outcome_df["crime_id"] == "89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a67755c99d4c32f25a0"]
rows.head()

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,outcome_type,year,month_num
1621642,89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a...,2024-10,Derbyshire Constabulary,Derbyshire Constabulary,-1.422896,53.234268,On or near Dixon's Road,E01033387,Chesterfield 010F,Suspect charged,2024,10
1621959,89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a...,2024-10,Derbyshire Constabulary,Derbyshire Constabulary,-1.422896,53.234268,On or near Dixon's Road,E01033387,Chesterfield 010F,Suspect charged,2024,10
1621964,89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a...,2024-10,Derbyshire Constabulary,Derbyshire Constabulary,-1.422896,53.234268,On or near Dixon's Road,E01033387,Chesterfield 010F,Suspect charged,2024,10
1621971,89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a...,2024-10,Derbyshire Constabulary,Derbyshire Constabulary,-1.422896,53.234268,On or near Dixon's Road,E01033387,Chesterfield 010F,Suspect charged,2024,10
1621976,89e5536fa5ed99ffc357ff2828e4d0cd04c531400ced7a...,2024-10,Derbyshire Constabulary,Derbyshire Constabulary,-1.422896,53.234268,On or near Dixon's Road,E01033387,Chesterfield 010F,Suspect charged,2024,10


determaning reson for repeats

In [13]:
rows.nunique()

crime_id        1
month           2
reported_by     1
falls_within    1
longitude       1
latitude        1
location        1
lsoa_code       1
lsoa_name       1
outcome_type    1
year            1
month_num       2
dtype: int64

the reason is time 

This code checks the dataset for missing values by calculating both the total number of null entries and the percentage of nulls in each column. street_df.isna().sum() counts how many missing values appear in every column, while street_df.isna().mean() * 100 converts those counts into percentages for easier interpretation. The results are then combined into a single DataFrame showing null_count and null_percent side by side, providing a clear overview of data completeness and helping identify which fields may require cleaning or imputation.

In [14]:
null_counts = street_df.isna().sum()
null_percent = (street_df.isna().mean() * 100).round(2)

pd.DataFrame({
    "null_count": null_counts,
    "null_percent": null_percent
})

,null_count,null_percent
crime_id,594430,16.28
month,0,0.00
reported_by,0,0.00
falls_within,0,0.00
longitude,87395,2.39
latitude,87395,2.39
location,0,0.00
lsoa_code,87407,2.39
lsoa_name,87407,2.39
crime_type,0,0.00


In [33]:
null_counts = outcome_df.isna().sum()
null_percent = (outcome_df.isna().mean() * 100).round(2)

pd.DataFrame({
    "null_count": null_counts,
    "null_percent": null_percent
})

,null_count,null_percent
crime_id,0,0.00
month,0,0.00
reported_by,0,0.00
falls_within,0,0.00
longitude,57768,2.15
latitude,57768,2.15
location,0,0.00
lsoa_code,57772,2.15
lsoa_name,57772,2.15
outcome_type,0,0.00


This step removes any rows where crime_id is missing and then drops the context column entirely, as this field is completely blank in the raw dataset and provides no useful information for analysis. By filtering out null crime_id values and removing an empty column, the dataset becomes cleaner and more efficient to work with. Running street_df.isna().sum() afterwards confirms that no unexpected missing values remain and that the dataset is ready for further processing.

In [15]:
street_df = street_df[street_df["crime_id"].notna()]
street_df = street_df.drop(columns=["context"])
street_df.isna().sum()

crime_id                     0
month                        0
reported_by                  0
falls_within                 0
longitude                87117
latitude                 87117
location                     0
lsoa_code                87126
lsoa_name                87126
crime_type                   0
last_outcome_category        0
year                         0
month_num                    0
dtype: int64

Some location fields in the street‑level crime data contain null values, and leaving these in the dataset is acceptable because the UK Police Service intentionally removes or anonymises location information for certain sensitive incidents. These missing values do not indicate an error in the data but reflect privacy protection built into the reporting process. Since the analysis in this project focuses on aggregated patterns (such as crime counts by force, month, or type) rather than precise point‑level mapping, the absence of some location coordinates does not affect the validity of the results and can safely remain in the dataset.

removing all rows that are exact duplicates

In [16]:
#remove exact dulicate rows street

before = len(street_df)


street_df = street_df.drop_duplicates()


after = len(street_df)



removed = before - after



#remove exact dulicate rows outcome

before2 = len(outcome_df)


outcome_df = outcome_df.drop_duplicates()


after2 = len(outcome_df)



removed2 = before2 - after2

print(before, removed, after)
print(before2, removed2, after2)


3056894 2621 3054273
2920000 236467 2683533


These conversions ensure that each column in the street‑level and outcomes dataset is stored in the most appropriate and reliable format for analysis. crime_id and month are converted to strings because they are identifiers rather than numeric values, meaning they should never be used for mathematical operations. The year and month_num fields are stored as integers so they can be sorted, filtered, or grouped correctly during time‑based analysis. Longitude and latitude are converted to floats to guarantee accurate handling of geographic coordinates. Setting these data types explicitly helps prevent errors later in the workflow and ensures consistent, predictable behaviour when merging, filtering, or visualising the data.

In [21]:

street_df["crime_id"] = street_df["crime_id"].astype(str)


street_df["year"] = street_df["year"].astype(int)
street_df["month_num"] = street_df["month_num"].astype(int)


street_df["longitude"] = street_df["longitude"].astype(float)
street_df["latitude"] = street_df["latitude"].astype(float)


street_df["month"] = street_df["month"].astype(str)
street_df.dtypes


outcome_df["crime_id"] = outcome_df["crime_id"].astype(str)


if "year" in outcome_df.columns:
    outcome_df["year"] = outcome_df["year"].astype(int)

if "month_num" in outcome_df.columns:
    outcome_df["month_num"] = outcome_df["month_num"].astype(int)


if "month" in outcome_df.columns:
    outcome_df["month"] = outcome_df["month"].astype(str)

In [19]:
street_df.dtypes

crime_id                  object
month                     object
reported_by               object
falls_within              object
longitude                float64
latitude                 float64
location                  object
lsoa_code                 object
lsoa_name                 object
crime_type                object
last_outcome_category     object
year                       int64
month_num                  int64
dtype: object

In [20]:
outcome_df.dtypes

crime_id         object
month            object
reported_by      object
falls_within     object
longitude       float64
latitude        float64
location         object
lsoa_code        object
lsoa_name        object
outcome_type     object
year              int64
month_num         int64
dtype: object

In [23]:
street_clean_df = (
    street_df
    .sort_values(["crime_id", "year", "month_num"])
    .drop_duplicates(subset="crime_id", keep="first")
)
outcome_clean_df = (
    outcome_df
    .sort_values(["crime_id", "year", "month_num"])
    .drop_duplicates(subset="crime_id", keep="last")
)


In [25]:
dupe_counts3 = (
    outcome_clean_df["crime_id"]
    .value_counts()
    .sort_values(ascending=False)
)
dupe_counts3
dupe_counts4 = (
    street_clean_df["crime_id"]
    .value_counts()
    .sort_values(ascending=False)
)
dupe_counts4

crime_id
0000067195b659bd3f0b70409a59cd11816770401ea3667b36d4ee4737216506    1
00002e4d642b2bfc486d795c731dadf763ec3e46a8cc47ae9835ca4eab19f936    1
000031623a52997b009b2914f1e90b620dd90251092899f9c85421c70404a614    1
0000150063e892e027db2aacd0af873ed5ae0c2aba0adb145fe7fbbd17225569    1
000017f6085ba12e398c78388b48ddd92cd921e909f1db3025b1ba75a9e1a7cc    1
                                                                   ..
ffffb96cdeaa672d2da6d0fd9a1712d46ae39db3e6b3179741ecdeb0355659bc    1
ffffce0b4956eecde6da31d2745740af3d5731b78a081f0ed02c1c55f655e450    1
ffffd114c5be58e434396d6d3be6d5dc04031c32b93072342d76487b99a74a8a    1
ffffd66fe0bf3f131626fcaf486d4858b9265ec87391c5216809014bc8204cd7    1
fffffcbcd983ef563271d168199fa92d0203ca7de72148f92590e4ad9980e851    1
Name: count, Length: 2617845, dtype: int64

In [26]:
dupe_counts4 = (
    street_clean_df["crime_id"]
    .value_counts()
    .sort_values(ascending=False)
)
dupe_counts4

crime_id
000004a8bc59a7595f94d35c8b02633eb0dd8077305e28930c42d012758c6833    1
000021067eb4a48db06e68e86d5d34539499c1da603afea6f49408ff341081f4    1
00008afbb5ab4cf29c674cebfe8cc46ca45896f9a7f5b8d47c93d69540eba437    1
00007bd4b2b88f28d720dc54a23e3798d105e579a33507d9e362e28281c48f63    1
000079800659fb4548f91ab28192f55c7bfb94f7ac7be749adbbf81d72cbcbcd    1
                                                                   ..
ffffd66fe0bf3f131626fcaf486d4858b9265ec87391c5216809014bc8204cd7    1
ffffdb28b2b023abca80e49802e57fbcf03b90d6dc36d865a221adcb8837e034    1
ffffdb54b18a4368103bdf710c7e90b4338eb9ca0100c7d5a9a92d54125a90d5    1
ffffdb90f385f1e54fd05ac941625a4e9c4ebc14c1274fafa9aa8068f5d43658    1
fffffcbcd983ef563271d168199fa92d0203ca7de72148f92590e4ad9980e851    1
Name: count, Length: 3040737, dtype: int64

This step compares the crime_id values in the street‑level dataset with those in the outcome dataset to identify any records that exist in one file but not the other. Crime IDs found only in the street data represent incidents that have no recorded outcome, which is expected because many crimes never progress to an outcome stage. Conversely, crime IDs found only in the outcome data usually reflect older or updated records that no longer appear in the current street‑level extract. Counting these differences helps verify the consistency of the two datasets and ensures that any later merge operation behaves as expected.

In [27]:
# crime_ids in street but not in outcomes
street_only = set(street_df["crime_id"]) - set(outcome_df["crime_id"])

# crime_ids in outcomes but not in street
outcome_only = set(outcome_df["crime_id"]) - set(street_df["crime_id"])

# Show counts
len(street_only), len(outcome_only)

(425646, 2754)

In [37]:
merged_df = street_clean_df.merge(
    outcome_clean_df[["crime_id", "outcome_type"]],
    on="crime_id",
    how="left"
)
len(street_clean_df), len(merged_df)

(3040737, 3040737)

verifng the merge

In [29]:
merged_df["outcome_type"].isna().sum()

np.int64(425646)

In [30]:
merged_df["outcome_type"].notna().sum()

np.int64(2615091)

In [31]:
merged_df[
    (merged_df["latitude"] < 49) |
    (merged_df["latitude"] > 61) |
    (merged_df["longitude"] < -8) |
    (merged_df["longitude"] > 2)
].shape

(0, 14)

exporting clean csvs

In [32]:
street_clean_df.to_csv("street_clean.csv", index=False)
outcome_clean_df.to_csv("outcome_clean.csv", index=False)
merged_df.to_csv("merged_clean.csv", index=False)

In [35]:
merged_df.head()

,crime_id,month,reported_by,falls_within,longitude,latitude,location,lsoa_code,lsoa_name,crime_type,last_outcome_category,year,month_num,outcome_type
0,000004a8bc59a7595f94d35c8b02633eb0dd8077305e28...,2023-07,Lancashire Constabulary,Lancashire Constabulary,-2.503001,53.752564,On or near Leamington Road,E01012588,Blackburn with Darwen 002E,Violence and sexual offences,Status update unavailable,2023,7,NaN
1,0000067195b659bd3f0b70409a59cd11816770401ea366...,2025-12,Cambridgeshire Constabulary,Cambridgeshire Constabulary,0.113614,52.173247,On or near Anstey Way,E01035526,Cambridge 015C,Criminal damage and arson,Investigation complete; no suspect identified,2025,12,Investigation complete; no suspect identified
2,000006a0d6919a710d3bd0a37d0ff31a0eb99ee5a1c20f...,2025-11,Norfolk Constabulary,Norfolk Constabulary,1.213493,52.649220,On or near Chestnut Close,E01026925,South Norfolk 017D,Violence and sexual offences,Unable to prosecute suspect,2025,11,Unable to prosecute suspect
3,000006d37e5a2ba94f983d908f144b72005d3d4786441e...,2026-01,Hampshire Constabulary,Hampshire Constabulary,-1.578626,50.922591,On or near Barleycorn Walk,E01022978,New Forest 006B,Violence and sexual offences,Under investigation,2026,1,NaN
4,0000150063e892e027db2aacd0af873ed5ae0c2aba0adb...,2023-09,Avon and Somerset Constabulary,Avon and Somerset Constabulary,NaN,NaN,No Location,NaN,NaN,Violence and sexual offences,Unable to prosecute suspect,2023,9,Unable to prosecute suspect
